# Step 1 : Data Understanding and Cleaning

The goal of this step is to understand the dataset, identify data-quality problems, and justify every cleaning decision.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load the dataset

In [ ]:
df = pd.read_csv("Food_delivery.csv")

df.head()

## 2. Dataset dimensions

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

## 3. General information and data types

In [ ]:
df.info()
df.columns

### Initial observation

The dataset contains numerical, categorical, identifier, and target variables.

- `Order_ID`: delivery identifier
- `Delivery_Time_min`: prediction target

Numerical explanatory variables:
- `Distance_km`
- `Preparation_Time_min`
- `Courier_Experience_yrs`

Categorical explanatory variables:
- `Weather`
- `Traffic_Level`
- `Time_of_Day`
- `Vehicle_Type`

## 4. Missing values

In [ ]:
df.isnull().sum()

In [ ]:
missing_data = pd.DataFrame({
    "Missing_values": df.isnull().sum(),
    "Percentage": df.isnull().mean() * 100
})

missing_data

### Missing-value decisions

The following columns contain missing values:

- `Weather`
- `Traffic_Level`
- `Time_of_Day`
- `Courier_Experience_yrs`

The categorical variables will later be imputed using the most frequent value.

`Courier_Experience_yrs` will later be imputed using the median.

The imputation itself will be performed inside the preprocessing pipeline after the train/test split in order to avoid data leakage.

## 5. Inspect categorical values

In [ ]:
categorical_columns = [
    "Weather",
    "Traffic_Level",
    "Time_of_Day",
    "Vehicle_Type"
]

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))

### Observation

The categorical labels are consistent and no parasitic text or inconsistent spelling was detected.

For example, no duplicated category forms such as `Rainy`, `rainy`, `RAINY`, or ` Rainy ` were found.

## 6. Numerical descriptive statistics

In [ ]:
numerical_columns = [
    "Distance_km",
    "Preparation_Time_min",
    "Courier_Experience_yrs",
    "Delivery_Time_min"
]

df[numerical_columns].describe()

## 7. Duplicate verification

In [ ]:
print("Duplicated rows:", df.duplicated().sum())

print(
    "Duplicated Order_ID:",
    df["Order_ID"].duplicated().sum()
)

### Duplicate decision

No duplicate rows or duplicate order identifiers were detected, so no records need to be removed.

## 8. Check impossible numerical values

In [ ]:
print(
    "Invalid distance:",
    (df["Distance_km"] <= 0).sum()
)

print(
    "Invalid preparation time:",
    (df["Preparation_Time_min"] <= 0).sum()
)

print(
    "Invalid courier experience:",
    (df["Courier_Experience_yrs"] < 0).sum()
)

print(
    "Invalid delivery time:",
    (df["Delivery_Time_min"] <= 0).sum()
)

### Numerical validity decision

No impossible numerical values were detected.

- Distance values are positive.
- Preparation times are positive.
- Courier experience is non-negative.
- Delivery times are positive.

## 9. Outlier detection with the IQR method

In [ ]:
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print("Column:", column)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Number of outliers:", len(outliers))

    return outliers

In [ ]:
detect_outliers(df, "Distance_km")

In [ ]:
detect_outliers(df, "Preparation_Time_min")

In [ ]:
detect_outliers(df, "Courier_Experience_yrs")

In [ ]:
delivery_outliers = detect_outliers(
    df,
    "Delivery_Time_min"
)

delivery_outliers

### Outlier decision

`Distance_km`, `Preparation_Time_min`, and `Courier_Experience_yrs` do not contain problematic statistical outliers.

`Delivery_Time_min` contains a small number of high statistical outliers.

These observations are retained because they remain operationally plausible and may represent genuine delayed deliveries.

Since detecting situations at risk of delay is part of the business objective, removing these cases could remove useful information.

## 10. Final cleaning decisions

| Column | Decision |
|---|---|
| `Order_ID` | Keep for traceability, exclude from model training |
| `Distance_km` | Keep as is |
| `Weather` | Impute missing values using most frequent category |
| `Traffic_Level` | Impute missing values using most frequent category |
| `Time_of_Day` | Impute missing values using most frequent category |
| `Vehicle_Type` | Keep as is |
| `Preparation_Time_min` | Keep as is |
| `Courier_Experience_yrs` | Impute missing values using median |
| `Delivery_Time_min` | Keep all plausible values, including high-delay outliers |

### Step 1 conclusion

The dataset contains 1,000 delivery observations and 9 variables.

Missing values were identified in `Weather`, `Traffic_Level`, `Time_of_Day`, and `Courier_Experience_yrs`.

Their treatment has been defined but will be implemented later inside a scikit-learn preprocessing pipeline to avoid data leakage.

No duplicate rows, duplicated order identifiers, inconsistent categorical labels, or impossible numerical values were detected.

`Order_ID` will not be used as a predictive feature because it is only an identifier.